# Comprehensive Rooftop Classification & Ablation Master Notebook
This notebook implements the entire Project Pipeline start-to-finish.
It loads the pure human-verified dataset, trains the ResNet baseline, visualizes 10 visual sample evaluations, and runs the highly massive 6-Part Architectural Ablation Study directly on the GPU.

In [ ]:
import os, json, time, warnings, copy
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42

CROPS_DIR = 'Rooftop_Crops'  # Make sure this is in your left sidebar or unzipped
LABELS_FILE = os.path.join(CROPS_DIR, 'labels.json')

CLASS_NAMES = ['flat', 'gable', 'hip']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Hardware Compute Device: {DEVICE}")

## 1. Datasets & Advanced Augmentation Strategies

In [ ]:
class RooftopDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

def load_data():
    with open(LABELS_FILE, 'r') as f:
        labels_dict = json.load(f)
    paths, labels = [], []
    for crop_name, label in labels_dict.items():
        if label in CLASS_TO_IDX: # Ignore skip labels, ensuring 100% human-verified purity
            p = os.path.join(CROPS_DIR, crop_name)
            if os.path.exists(p):
                paths.append(p)
                labels.append(CLASS_TO_IDX[label])
    return paths, labels

def get_transforms(img_size, augment_level='standard'):
    normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    if augment_level == 'none':
        train_tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), normalize])
    elif augment_level == 'light':
        train_tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize])
    elif augment_level == 'standard':
        train_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(), normalize
        ])
    elif augment_level == 'heavy':
        train_tf = transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)), transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(180),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            transforms.RandomGrayscale(p=0.1), transforms.GaussianBlur(3, sigma=(0.1, 2.0)),
            transforms.ToTensor(), normalize
        ])
    val_tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor(), normalize])
    return train_tf, val_tf

def make_loaders(train_paths, train_labels, val_paths, val_labels, train_tf, val_tf, batch_size=32, use_sampler=True):
    train_ds = RooftopDataset(train_paths, train_labels, transform=train_tf)
    val_ds = RooftopDataset(val_paths, val_labels, transform=val_tf)
    if use_sampler:
        counts = Counter(train_labels)
        weights = [1.0 / counts[l] for l in train_labels]
        sampler = WeightedRandomSampler(weights, len(weights))
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader

## 2. Deep Learning Neural Network Definitions

In [ ]:
def build_model(arch, freeze_strategy='partial', num_classes=3):
    if arch == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
        freeze_keywords = {'partial': ['layer3', 'layer4', 'fc'], 'full': ['fc']}
    elif arch == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, num_classes))
        freeze_keywords = {'partial': ['layer3', 'layer4', 'fc'], 'full': ['fc']}
    elif arch == 'mobilenet_v3_small':
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
        freeze_keywords = {'partial': ['features.9', 'features.10', 'features.11', 'classifier'], 'full': ['classifier']}
    elif arch == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier.in_features, num_classes))
        freeze_keywords = {'partial': ['features.denseblock4', 'features.norm5', 'classifier'], 'full': ['classifier']}

    if freeze_strategy != 'none' and freeze_strategy in freeze_keywords:
        kw = freeze_keywords[freeze_strategy]
        for name, param in model.named_parameters():
            if not any(k in name for k in kw):
                param.requires_grad = False
    return model

## 3. Training Loop Engine with Real-Time Progress Tracking

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, num_epochs=20, lr=1e-4):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0; best_state = None
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            train_pbar.set_postfix(loss=(running_loss/total), acc=(correct/total))

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        running_loss, correct, total = 0, 0, 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
        with torch.no_grad():
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss = running_loss / total
        val_acc = correct / total
        scheduler.step(val_acc)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    if best_state: model.load_state_dict(best_state)
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images.to(DEVICE))
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.to(DEVICE).cpu().numpy())

    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    per_class = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)
    
    metrics = {
        'best_val_acc': best_val_acc, 'f1': f1, 'precision': prec, 'recall': rec,
        'per_class_f1': per_class[2].tolist(),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad)
    }
    return history, metrics, model

def run_experiment(name, model, train_loader, val_loader, num_epochs=15):
    t0 = time.time()
    history, metrics, trained_model = train_and_evaluate(model, train_loader, val_loader, num_epochs)
    elapsed = time.time() - t0
    print(f"{name:<30} -> Acc={metrics['best_val_acc']:.4f} | F1={metrics['f1']:.4f} | Param={metrics['trainable_params']/1e6:.2f}M | {elapsed:.1f}s")
    
    return {'metrics': metrics, 'time': elapsed, 'name': name}, trained_model

## 4. Baseline Model Training & Visual 10-Image Test
Train the baseline ResNet18 model on the fully human verified 100% dataset, and visualize the output on exactly 10 validation images.

In [ ]:
all_paths, all_labels = load_data()
print(f"Total 100% Human-Verified Samples: {len(all_labels)}")
train_p, val_p, train_l, val_l = train_test_split(all_paths, all_labels, test_size=0.2, random_state=RANDOM_SEED, stratify=all_labels)

trn_tf, val_tf = get_transforms(160, 'heavy')
trn_loader, v_loader = make_loaders(train_p, train_l, val_p, val_l, trn_tf, val_tf, use_sampler=True)

res18_model = build_model('resnet18', 'partial')
print("\n--- Training Core Baseline ResNet18 (15 Epochs) ---")
res18_results, res18_trained = run_experiment("Baseline_Resnet18", res18_model, trn_loader, v_loader, num_epochs=50)

# Visualize exactly 10 images
res18_trained.eval()
indices = random.sample(range(len(val_p)), 10)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

with torch.no_grad():
    for i, idx in enumerate(indices):
        img_path = val_p[idx]
        true_label = CLASS_NAMES[val_l[idx]]
        raw_img = Image.open(img_path).convert('RGB')
        tensor_img = val_tf(raw_img).unsqueeze(0).to(DEVICE)
        
        out = res18_trained(tensor_img)
        _, pred_idx = out.max(1)
        pred_label = CLASS_NAMES[pred_idx.item()]
        
        ax = axes[i]
        ax.imshow(raw_img)
        color = 'green' if true_label == pred_label else 'red'
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 5. The Massive 6-Part Architectural Ablation Study
Now we scientifically validate every structural decision we made during the engineering process pipeline. This block will meticulously iterate over massive structural changes in Neural Topologies, Data Manipulations, Model Freezing scales, Resizing variables, and Data Volume restrictions.

In [ ]:
print("=" * 60)
print("STARTING COMPREHENSIVE ABLATION LOOP (GPU ONLY)")
print("=" * 60)

ABLATION_EPOCHS = 20
all_results = {}

# --- EXPERIMENT 1: ARCHITECTURES ---
print("\n[PART 1/6] Running Neural Architectures...")
for arch in ['resnet18', 'mobilenet_v3_small', 'densenet121', 'resnet50']:
    model = build_model(arch, 'partial')
    res, _ = run_experiment(f"arch_{arch}", model, trn_loader, v_loader, ABLATION_EPOCHS)
    all_results[f"arch_{arch}"] = res

# --- EXPERIMENT 2: AUGMENTATION LEVEL ---
print("\n[PART 2/6] Running Augmentation Limits on ResNet18...")
for aug in ['none', 'light', 'standard', 'heavy']:
    tf_tr, tf_v = get_transforms(160, aug)
    tl, vl = make_loaders(train_p, train_l, val_p, val_l, tf_tr, tf_v)
    model = build_model('resnet18', 'partial')
    res, _ = run_experiment(f"aug_{aug}", model, tl, vl, ABLATION_EPOCHS)
    all_results[f"aug_{aug}"] = res

# --- EXPERIMENT 3: NETWORK FREEZING ---
print("\n[PART 3/6] Running Layer Freeze Constraints...")
for freeze in ['none', 'partial', 'full']:
    model = build_model('resnet18', freeze)
    res, _ = run_experiment(f"freeze_{freeze}", model, trn_loader, v_loader, ABLATION_EPOCHS)
    all_results[f"freeze_{freeze}"] = res

# --- EXPERIMENT 4: IMAGE SIZING ---
print("\n[PART 4/6] Running Geometric Tensor Size Sweeps (64->224)...")
for sz in [64, 96, 128, 160, 224]:
    tf_tr, tf_v = get_transforms(sz, 'standard')
    tl, vl = make_loaders(train_p, train_l, val_p, val_l, tf_tr, tf_v)
    model = build_model('resnet18', 'partial')
    res, _ = run_experiment(f"imgsize_{sz}", model, tl, vl, ABLATION_EPOCHS)
    all_results[f"imgsize_{sz}"] = res

# --- EXPERIMENT 5: DATA SIZE ---
print("\n[PART 5/6] Running Training Data Volume Thresholds...")
for pct in [25, 50, 75, 100]:
    if pct < 100:
        n = int(len(train_p) * pct / 100)
        sub_p, _, sub_l, _ = train_test_split(train_p, train_l, train_size=n, random_state=RANDOM_SEED, stratify=train_l)
    else:
        sub_p, sub_l = train_p, train_l
    
    tl, vl = make_loaders(sub_p, sub_l, val_p, val_l, trn_tf, v_loader.dataset.transform) # Using standard transforms
    model = build_model('resnet18', 'partial')
    res, _ = run_experiment(f"datasize_{pct}pct", model, tl, vl, ABLATION_EPOCHS)
    all_results[f"datasize_{pct}pct"] = res

# --- EXPERIMENT 6: SAMPLER ABLATION ---
print("\n[PART 6/6] Weighted Imbalance Correctors...")
for use_sampler, lbl in [(True, 'weighted'), (False, 'unweighted')]:
    tl, vl = make_loaders(train_p, train_l, val_p, val_l, trn_tf, v_loader.dataset.transform, use_sampler=use_sampler)
    model = build_model('resnet18', 'partial')
    res, _ = run_experiment(f"sampler_{lbl}", model, tl, vl, ABLATION_EPOCHS)
    all_results[f"sampler_{lbl}"] = res

print("\n\nAll Ablation Computations Successfully Finished!")

## 6. Plotting the Findings
Generating the final architecture vs efficiency correlation comparison graph.

In [ ]:
arch_results = {k: v for k, v in all_results.items() if k.startswith('arch_')}
names = [k.replace('arch_', '') for k in arch_results.keys()]
accs = [v['metrics']['best_val_acc'] for v in arch_results.values()]
params = [v['metrics']['trainable_params'] / 1e6 for v in arch_results.values()]

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Network Architecture', fontweight='bold')
ax1.set_ylabel('Validation Accuracy (%)', color=color, fontweight='bold')
bars = ax1.bar(names, [a*100 for a in accs], color=color, alpha=0.7)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim([0, 100])

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Million Parameters (Compute Weight)', color=color, fontweight='bold')
ax2.plot(names, params, color=color, marker='o', linewidth=3, markersize=10)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Final Study Graph: Accuracy vs Artificial Brain Size', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

print("=" * 60)
print("FINAL PIPELINE END-TO-END COMPLETION")
print("=" * 60)